In [ ]:
!pip install -q music21 midi2audio
!apt-get -qq install -y fluidsynth > /dev/null

In [ ]:
import os
import re
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from music21 import converter
from midi2audio import FluidSynth
from IPython.display import Audio, display

keras.utils.set_random_seed(42)
print(tf.__version__, tf.config.list_physical_devices("GPU"))

In [ ]:
songs_url = "https://raw.githubusercontent.com/aamini/introtodeeplearning/master/mitdeeplearning/data/irish.abc"
urllib.request.urlretrieve(songs_url, "irish.abc")

with open("irish.abc", "r", encoding="utf-8") as song_file:
    raw_text = song_file.read()

songs = [song.strip() for song in re.split(r"\n\s*\n", raw_text) if "K:" in song and len(song.strip()) > 50]
print("songs found:", len(songs))
print()
print(songs[0])

In [ ]:
song_lengths = [len(song) for song in songs]
plt.figure(figsize=(8, 4))
plt.hist(song_lengths, bins=40, color="steelblue")
plt.xlabel("characters per song")
plt.ylabel("number of songs")
plt.title("Song length distribution")
plt.show()

In [ ]:
all_songs_text = "\n\n".join(songs)
vocabulary = sorted(set(all_songs_text))
char_to_index = {char: index for index, char in enumerate(vocabulary)}
index_to_char = np.array(vocabulary)
encoded_text = np.array([char_to_index[char] for char in all_songs_text], dtype=np.int32)

print("total characters:", len(all_songs_text))
print("unique characters:", len(vocabulary))
print(repr(all_songs_text[:60]))
print(encoded_text[:60])

In [ ]:
sequence_length = 100
batch_size = 64
window_step = 3

def make_windows(encoded, step):
    start_positions = np.arange(0, len(encoded) - sequence_length - 1, step)
    inputs = np.stack([encoded[start:start + sequence_length] for start in start_positions])
    targets = np.stack([encoded[start + 1:start + sequence_length + 1] for start in start_positions])
    return inputs, targets

split_point = int(len(encoded_text) * 0.9)
train_inputs, train_targets = make_windows(encoded_text[:split_point], window_step)
val_inputs, val_targets = make_windows(encoded_text[split_point:], sequence_length)

train_dataset = tf.data.Dataset.from_tensor_slices((train_inputs, train_targets)).shuffle(20000).batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
val_dataset = tf.data.Dataset.from_tensor_slices((val_inputs, val_targets)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

print("train windows:", train_inputs.shape, "validation windows:", val_inputs.shape)
print("input :", repr("".join(index_to_char[train_inputs[0][:40]])))
print("target:", repr("".join(index_to_char[train_targets[0][:40]])))

In [ ]:
embedding_dim = 256
lstm_units = 1024

music_model = keras.Sequential([
    layers.Input(shape=(None,)),
    layers.Embedding(len(vocabulary), embedding_dim),
    layers.LSTM(lstm_units, return_sequences=True),
    layers.Dropout(0.2),
    layers.Dense(len(vocabulary))
])

music_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=2e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)
music_model.summary()

In [ ]:
training_callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)
]
history = music_model.fit(train_dataset, validation_data=val_dataset, epochs=25, callbacks=training_callbacks)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set_title("Cross-entropy loss")
axes[0].set_xlabel("epoch")
axes[0].legend()
axes[1].plot(history.history["accuracy"], label="train")
axes[1].plot(history.history["val_accuracy"], label="validation")
axes[1].set_title("Next character accuracy")
axes[1].set_xlabel("epoch")
axes[1].legend()
plt.show()

In [ ]:
def generate_music(start_text="X:", generation_length=2000, temperature=0.8):
    generated_ids = [char_to_index[char] for char in start_text]
    for _ in range(generation_length):
        context = np.array([generated_ids[-sequence_length:]], dtype=np.int32)
        logits = music_model(context, training=False)[0, -1].numpy().astype(np.float64)
        logits = logits / temperature
        probabilities = np.exp(logits - logits.max())
        probabilities = probabilities / probabilities.sum()
        generated_ids.append(int(np.random.choice(len(vocabulary), p=probabilities)))
    return "".join(index_to_char[generated_ids])

def extract_songs(text):
    pieces = re.split(r"\n\s*\n", text)
    return [piece.strip() for piece in pieces if piece.strip().startswith("X:") and "K:" in piece and len(piece.strip()) > 80]

In [ ]:
for temperature in [0.4, 0.8, 1.2]:
    print(f"===== temperature {temperature} =====")
    print(generate_music(generation_length=300, temperature=temperature))
    print()

In [ ]:
generated_songs = []
attempt = 0
while len(generated_songs) < 3 and attempt < 6:
    attempt += 1
    generated_text = generate_music(temperature=0.8)
    generated_songs.extend(extract_songs(generated_text))

print("complete songs generated:", len(generated_songs))
for number, song in enumerate(generated_songs, start=1):
    print(f"--- song {number} ---")
    print(song)
    print()

In [ ]:
os.makedirs("generated_music", exist_ok=True)
midi_files = []

for number, song in enumerate(generated_songs, start=1):
    abc_path = f"generated_music/song_{number}.abc"
    midi_path = f"generated_music/song_{number}.mid"
    with open(abc_path, "w", encoding="utf-8") as abc_file:
        abc_file.write(song + "\n")
    try:
        score = converter.parse(abc_path)
        score.write("midi", fp=midi_path)
        midi_files.append(midi_path)
        print("saved", abc_path, "and", midi_path)
    except Exception as error:
        print(f"song {number} has invalid abc notation and was kept only as .abc: {error}")

In [ ]:
sound_font = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
synthesizer = FluidSynth(sound_font)

for midi_path in midi_files:
    wav_path = midi_path.replace(".mid", ".wav")
    try:
        synthesizer.midi_to_audio(midi_path, wav_path)
        print(wav_path)
        display(Audio(wav_path))
    except Exception as error:
        print(f"could not render {midi_path}: {error}")

In [ ]:
music_model.save("music_model.keras")
!zip -q -r generated_music.zip generated_music
print(sorted(os.listdir("generated_music")))